# Day 9 v3 — Silver: All Realtime Blob Entities (Job Parameters + Data Quality)

**Source:** `bronze-volume/realtime/<entity>/YYYY/MM/DD/HH/`  
**Sink:** `silver-volume/realtime/<entity>/` (Delta — full overwrite or MERGE)

All parameters **default to current UTC time** — no job configuration needed for regular hourly runs.  
Pass parameters explicitly only when doing a **backfill** for a specific partition.

| Parameter | Default | Example override |
|---|---|---|
| `load_type` | `incremental` | `full` |
| `ingestion_year` | current UTC year | `2026` |
| `ingestion_month` | current UTC month | `06` |
| `ingestion_day` | current UTC day | `01` |
| `ingestion_hour` | current UTC hour | `14` |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Parameters
# Only load_type is required from ADF / Databricks Job.
# Date/hour params default to the PREVIOUS UTC hour — current hour's files are still
# being written when the job triggers, so we always process the completed prior hour.
#
#   load_type       : "full" | "incremental"  (pass from ADF, default: incremental)
#   ingestion_year  : YYYY                    (default: previous UTC hour's year)
#   ingestion_month : MM                      (default: previous UTC hour's month)
#   ingestion_day   : DD                      (default: previous UTC hour's day)
#   ingestion_hour  : HH                      (default: previous UTC hour)

from datetime import timedelta
_now  = datetime.now(timezone.utc)
_prev = _now - timedelta(hours=1)   # always process the completed prior hour

def _get_param(key, default):
    try:
        val = dbutils.widgets.get(key).strip()
        return val if val else default
    except Exception:
        return default

LOAD_TYPE       = _get_param("load_type",       "incremental")
INGESTION_YEAR  = _get_param("ingestion_year",  str(_prev.year))
INGESTION_MONTH = _get_param("ingestion_month", f"{_prev.month:02d}")
INGESTION_DAY   = _get_param("ingestion_day",   f"{_prev.day:02d}")
INGESTION_HOUR  = _get_param("ingestion_hour",  f"{_prev.hour:02d}")

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")
print(f"ingestion_day   : {INGESTION_DAY}")
print(f"ingestion_hour  : {INGESTION_HOUR}")
print("Parameters resolved.")

In [ ]:
# Cell 3 — Constants
BRONZE_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime"
SILVER_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime"
QUARANTINE_BASE = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/quarantine/realtime"
PIPELINE_NAME   = "job_silver_blob_realtime_v3"
RUN_TS          = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"RUN_TS : {RUN_TS}")

In [ ]:
# Cell 4 — Entity schema registry
ENTITY_REGISTRY = {
    "charging_sessions": {
        "natural_key": "session_id",
        "cdc_field"  : "updated_at",
        "csv_schema" : StructType([
            StructField("session_id",          StringType(), True),
            StructField("charger_id",          StringType(), True),
            StructField("station_id",          StringType(), True),
            StructField("vehicle_id",          StringType(), True),
            StructField("customer_id",         StringType(), True),
            StructField("plug_in_ts",          StringType(), True),
            StructField("charge_end_ts",       StringType(), True),
            StructField("duration_min",        StringType(), True),
            StructField("energy_kwh",          StringType(), True),
            StructField("peak_kw",             StringType(), True),
            StructField("connector_type",      StringType(), True),
            StructField("session_status",      StringType(), True),
            StructField("tariff_id",           StringType(), True),
            StructField("raw_device_temp_c",   StringType(), True),
            StructField("signal_strength_dbm", StringType(), True),
            StructField("firmware_ver",        StringType(), True),
            StructField("state_code",          StringType(), True),
            StructField("protocol_version",    StringType(), True),
            StructField("ingested_at",         StringType(), True),
        ]),
        "ts_cols"      : ["plug_in_ts", "charge_end_ts", "ingested_at"],
        "numeric_casts": {
            "duration_min"       : "integer",
            "energy_kwh"         : "decimal(10,4)",
            "peak_kw"            : "decimal(10,4)",
            "raw_device_temp_c"  : "decimal(6,2)",
            "signal_strength_dbm": "integer",
        },
        "negative_check": ["energy_kwh", "peak_kw", "duration_min"],
    },
    "maintenance_events": {
        "natural_key": "event_id",
        "cdc_field"  : "updated_at",
        "csv_schema" : StructType([
            StructField("event_id",           StringType(), True),
            StructField("charger_id",         StringType(), True),
            StructField("station_id",         StringType(), True),
            StructField("employee_id",        StringType(), True),
            StructField("root_cause",         StringType(), True),
            StructField("mttr_hours",         StringType(), True),
            StructField("event_ts",           StringType(), True),
            StructField("device_error_code",  StringType(), True),
            StructField("voltage_at_fault",   StringType(), True),
            StructField("ambient_temp_c",     StringType(), True),
            StructField("state_code",         StringType(), True),
            StructField("part_replaced",      StringType(), True),
            StructField("raw_payload_bytes",  StringType(), True),
            StructField("ingested_at",        StringType(), True),
        ]),
        "ts_cols"      : ["event_ts", "ingested_at"],
        "numeric_casts": {
            "mttr_hours"       : "decimal(8,2)",
            "voltage_at_fault" : "decimal(8,2)",
            "ambient_temp_c"   : "decimal(6,2)",
            "raw_payload_bytes": "long",
        },
        "negative_check": ["mttr_hours"],
    },
}
print(f"Entity registry: {list(ENTITY_REGISTRY.keys())}")

In [ ]:
# Cell 5 — Helper functions

def _safe_ts(col_name):
    """ANSI-safe timestamp: try ISO string, fall back to Unix epoch float."""
    iso   = F.try_to_timestamp(F.col(col_name))
    epoch = F.try_to_timestamp(F.col(col_name).try_cast("double").cast("string"))
    return F.when(iso.isNotNull(), iso).otherwise(epoch).alias(col_name)

def folder_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def write_quarantine(df, entity_name, reason):
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta").mode("append").option("mergeSchema", "true")
          .save(f"{QUARANTINE_BASE}/{entity_name}")
    )

def get_bronze_paths(entity_name):
    base = f"{BRONZE_REALTIME}/{entity_name}"
    if not folder_exists(base):
        return []
    if LOAD_TYPE == "full":
        return [base]
    path = f"{base}/{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}"
    if INGESTION_HOUR:
        path = f"{path}/{INGESTION_HOUR}"
    return [path] if folder_exists(path) else []

print("Helpers defined")

In [ ]:
# Cell 6 — transform_entity() orchestrator

def transform_entity(entity_name, config):
    result = {"entity_name": entity_name, "status": "failed",
              "bronze_rows": 0, "null_pk": 0, "null_cdc": 0,
              "corrupt": 0, "negative": 0, "dups": 0, "silver_rows": 0, "error": None}
    try:
        natural_key  = config["natural_key"]
        cdc_field    = config["cdc_field"]
        csv_schema   = config["csv_schema"]
        ts_cols      = config["ts_cols"]
        numeric_cast = config["numeric_casts"]
        neg_cols     = config["negative_check"]
        silver_path  = f"{SILVER_REALTIME}/{entity_name}"
        csv_cols     = [f.name for f in csv_schema.fields]

        paths = get_bronze_paths(entity_name)
        if not paths:
            result["status"] = "skipped"
            result["error"]  = "No Bronze files found"
            return result

        # Read + derive updated_at
        raw_df = (
            spark.read
            .option("header", "true").option("recursiveFileLookup", "true")
            .schema(csv_schema).csv(*paths)
            .select(*[F.col(c) for c in csv_cols], F.col("_metadata.file_path").alias("_fp"))
        )
        result["bronze_rows"] = raw_df.count()

        raw_df = (
            raw_df
            .withColumn("_y", F.regexp_extract("_fp", r"/(\d{4})/", 1))
            .withColumn("_m", F.regexp_extract("_fp", r"/\d{4}/(\d{2})/", 1))
            .withColumn("_d", F.regexp_extract("_fp", r"/\d{4}/\d{2}/(\d{2})/", 1))
            .withColumn("_h", F.regexp_extract("_fp", r"/\d{4}/\d{2}/\d{2}/(\d{2})/", 1))
            .withColumn("updated_at", F.to_timestamp(
                F.concat_ws(" ", F.concat_ws("-", "_y", "_m", "_d"), "_h"), "yyyy-MM-dd HH"))
            .drop("_fp", "_y", "_m", "_d", "_h")
        )

        # Trim strings
        for c, t in raw_df.dtypes:
            if t == "string":
                raw_df = raw_df.withColumn(c, F.trim(F.col(c)))

        # NULL PK guard
        null_pk = raw_df.filter(F.col(natural_key).isNull() | (F.col(natural_key) == ""))
        result["null_pk"] = null_pk.count()
        write_quarantine(null_pk, entity_name, f"null_pk:{natural_key}")
        raw_df = raw_df.filter(F.col(natural_key).isNotNull() & (F.col(natural_key) != ""))

        # NULL CDC guard
        null_cdc = raw_df.filter(F.col(cdc_field).isNull())
        result["null_cdc"] = null_cdc.count()
        write_quarantine(null_cdc, entity_name, f"null_cdc:{cdc_field}")
        raw_df = raw_df.filter(F.col(cdc_field).isNotNull())

        # Cast timestamps (ANSI-safe)
        for c in ts_cols:
            if c in [col for col, _ in raw_df.dtypes]:
                iso   = F.try_to_timestamp(F.col(c))
                epoch = F.try_to_timestamp(F.col(c).try_cast("double").cast("string"))
                raw_df = raw_df.withColumn(c, F.when(iso.isNotNull(), iso).otherwise(epoch))

        # Cast numerics (ANSI-safe try_cast)
        for c, t in numeric_cast.items():
            if c in [col for col, _ in raw_df.dtypes]:
                raw_df = raw_df.withColumn(c, F.col(c).try_cast(t))

        # Negative value quarantine
        neg_cond = F.lit(False)
        for c in neg_cols:
            if c in [col for col, _ in raw_df.dtypes]:
                neg_cond = neg_cond | (F.col(c) < 0)
        neg_df = raw_df.filter(neg_cond)
        result["negative"] = neg_df.count()
        write_quarantine(neg_df, entity_name, "negative_value")
        raw_df = raw_df.filter(~neg_cond)

        # Audit + dedup
        audited_df = (
            raw_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit(LOAD_TYPE))
            .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
        )
        before = audited_df.count()
        window = Window.partitionBy(natural_key).orderBy(F.col(cdc_field).desc())
        deduped_df = (
            audited_df
            .withColumn("_rn", F.row_number().over(window))
            .filter(F.col("_rn") == 1).drop("_rn")
        )
        result["dups"] = before - deduped_df.count()

        # Write Silver
        if LOAD_TYPE == "full" or not folder_exists(silver_path):
            deduped_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)
        else:
            DeltaTable.forPath(spark, silver_path).alias("t") \
                .merge(deduped_df.alias("s"), f"t.{natural_key} = s.{natural_key}") \
                .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

        result["silver_rows"] = deduped_df.count()
        result["status"]      = "succeeded"

    except Exception as e:
        result["error"] = str(e)
    return result

print("transform_entity() defined")

In [ ]:
# Cell 7 — Run all entities
partition_label = f"{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}" + (f"/{INGESTION_HOUR}" if INGESTION_HOUR else "")
print(f"load_type={LOAD_TYPE} | partition={partition_label}")

results = []
for entity_name, config in ENTITY_REGISTRY.items():
    print(f"  {entity_name} ...", end=" ")
    r = transform_entity(entity_name, config)
    results.append(r)
    print(f"{r['status']} (bronze={r['bronze_rows']} -> silver={r['silver_rows']})" if r["status"] != "failed" else f"FAILED: {r['error']}")

In [ ]:
# Cell 8 — Run summary
succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 95)
print("SILVER BLOB REALTIME v3 — RUN SUMMARY")
print("=" * 95)
print(f"  load_type={LOAD_TYPE} | partition={partition_label} | run_ts={RUN_TS}")
print()
print(f"  {'ENTITY':<25} {'STATUS':<10} {'BRONZE':>7} {'NULL_PK':>7} {'NULL_CDC':>8} {'NEG':>5} {'DUPS':>6} {'SILVER':>7}")
print(f"  {'-'*25} {'-'*10} {'-'*7} {'-'*7} {'-'*8} {'-'*5} {'-'*6} {'-'*7}")
for r in results:
    print(f"  {r['entity_name']:<25} {r['status']:<10} {r['bronze_rows']:>7} {r['null_pk']:>7} {r['null_cdc']:>8} {r['negative']:>5} {r['dups']:>6} {r['silver_rows']:>7}")
    if r["error"]:
        print(f"    Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed.")
else:
    print(f"\nAll {len(succeeded)} entities written to Silver successfully.")

In [ ]:
# Cell 9 — Verify Silver + Quarantine
print("SILVER VERIFICATION")
for name in ENTITY_REGISTRY:
    path = f"{SILVER_REALTIME}/{name}"
    if folder_exists(path):
        df = spark.read.format("delta").load(path)
        print(f"  {name:<25} rows={df.count()}  cols={len(df.columns)}")
    else:
        print(f"  {name:<25} NOT FOUND")

print("\nQUARANTINE CHECK")
for name in ENTITY_REGISTRY:
    qp = f"{QUARANTINE_BASE}/{name}"
    if folder_exists(qp):
        qdf = spark.read.format("delta").load(qp)
        print(f"  {name:<25} quarantine rows={qdf.count()}")
        qdf.groupBy("reject_reason").count().show(truncate=False)
    else:
        print(f"  {name:<25} no quarantine rows")